In [3]:
# ============================================================
# ERA5-LAND REPRESENTATIVE COORDINATE SELECTION
# Pincode: 380006, Ahmedabad
# ============================================================

import ee
import json
import pandas as pd

# ------------------------------------------------------------
# 1. Initialize Google Earth Engine
# ------------------------------------------------------------

ee.Initialize(project="development-hruday")


# ------------------------------------------------------------
# 2. Load Pincode Boundary
# ------------------------------------------------------------

boundary_file = "../raw/boundary_file/380006_boundary.geojson"

with open(boundary_file, "r") as f:
    geojson_dict = json.load(f)

coords = geojson_dict["features"][0]["geometry"]["coordinates"]

roi = ee.Geometry.Polygon(coords)


# ------------------------------------------------------------
# 3. Calculate Pincode Centroid
# ------------------------------------------------------------

centroid = roi.centroid()

centroid_coords = centroid.coordinates().getInfo()

centroid_lon = centroid_coords[0]
centroid_lat = centroid_coords[1]

print("Pincode centroid:")
print("Longitude:", centroid_lon)
print("Latitude :", centroid_lat)


# ------------------------------------------------------------
# 4. Load ERA5-Land Dataset
# ------------------------------------------------------------

era5 = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .filterDate("2005-01-01", "2005-01-02")
    .select([
        "temperature_2m",
        "surface_solar_radiation_downwards_sum",
        "total_precipitation_sum"
    ])
)


# ------------------------------------------------------------
# 5. Create a Point from the Pincode Centroid
# ------------------------------------------------------------

representative_point = ee.Geometry.Point(
    [centroid_lon, centroid_lat]
)


# ------------------------------------------------------------
# 6. Find the ERA5-Land Pixel Containing the Centroid
# ------------------------------------------------------------

# Take one ERA5-Land image.
# The ERA5-Land grid does not change spatially over time,
# so one image is sufficient to determine the grid location.

image = ee.Image(era5.first())


# ------------------------------------------------------------
# 7. Get the Pixel's Actual Grid-Cell Center Coordinates
# ------------------------------------------------------------

projection = image.projection()

# Transform the centroid into the ERA5-Land grid.
# This identifies the pixel containing the centroid.

pixel_coordinates = (
    representative_point
    .transform(projection, 1)
    .coordinates()
    .getInfo()
)

print("\nCentroid in ERA5 projection:")
print(pixel_coordinates)


# ------------------------------------------------------------
# 8. Create a Pixel Geometry Around the Centroid
# ------------------------------------------------------------

# Reproject the centroid to the ERA5 grid and get the
# corresponding pixel footprint.

pixel_geometry = (
    image
    .select("temperature_2m")
    .sample(
        region=representative_point,
        scale=11132,
        geometries=True
    )
)


# ------------------------------------------------------------
# 9. Retrieve the Actual Pixel Feature
# ------------------------------------------------------------

pixel_info = pixel_geometry.first().getInfo()

print("\nERA5 pixel information:")
print(pixel_info)


# ------------------------------------------------------------
# 10. Extract Actual ERA5 Pixel-Center Coordinates
# ------------------------------------------------------------

pixel_lon = pixel_info["geometry"]["coordinates"][0]
pixel_lat = pixel_info["geometry"]["coordinates"][1]

print("\nRepresentative ERA5-Land pixel:")
print("Longitude:", pixel_lon)
print("Latitude :", pixel_lat)


# ------------------------------------------------------------
# 11. Verify the Pixel by Extracting a Climate Variable
# ------------------------------------------------------------

temperature_value = (
    image
    .reduceRegion(
        reducer=ee.Reducer.first(),
        geometry=representative_point,
        scale=11132
    )
    .get("temperature_2m")
    .getInfo()
)

print("\nTemperature on 2005-01-01:")
print("Kelvin:", temperature_value)

if temperature_value is not None:
    print("Celsius:", temperature_value - 273.15)


# ------------------------------------------------------------
# 12. Final Coordinate Variables
# ------------------------------------------------------------

REPRESENTATIVE_LON = pixel_lon
REPRESENTATIVE_LAT = pixel_lat

print("\n==========================================")
print("FINAL REPRESENTATIVE COORDINATES")
print("==========================================")
print(f"Latitude : {REPRESENTATIVE_LAT}")
print(f"Longitude: {REPRESENTATIVE_LON}")

Pincode centroid:
Longitude: 72.56121992376342
Latitude : 23.02227361979656

Centroid in ERA5 projection:
[2526.1121992376343, 670.2772638020342]

ERA5 pixel information:
{'type': 'Feature', 'geometry': {'geodesic': False, 'type': 'Point', 'coordinates': [72.60115569230976, 22.999693294401865]}, 'id': '0', 'properties': {'temperature_2m': 290.8949197133382}}

Representative ERA5-Land pixel:
Longitude: 72.60115569230976
Latitude : 22.999693294401865

Temperature on 2005-01-01:
Kelvin: 290.8949197133382
Celsius: 17.74491971333822

FINAL REPRESENTATIVE COORDINATES
Latitude : 22.999693294401865
Longitude: 72.60115569230976


In [4]:
import ee
import json

ee.Initialize(project="development-hruday")

# Load pincode boundary
with open("../raw/boundary_file/380006_boundary.geojson", "r") as f:
    geojson_dict = json.load(f)

coords = geojson_dict["features"][0]["geometry"]["coordinates"]
roi = ee.Geometry.Polygon(coords)

# Calculate centroid
centroid = roi.centroid()
centroid_coords = centroid.coordinates().getInfo()

centroid_lon = centroid_coords[0]
centroid_lat = centroid_coords[1]

print("Pincode centroid:")
print(centroid_lat, centroid_lon)


# Load ERA5-Land
image = (
    ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
    .filterDate("2005-01-01", "2005-01-02")
    .first()
)


# Find the ERA5-Land pixel containing the centroid
sample = (
    image
    .select("temperature_2m")
    .sample(
        region=ee.Geometry.Point([centroid_lon, centroid_lat]),
        scale=11132,
        geometries=True
    )
    .first()
    .getInfo()
)


# Extract actual ERA5 grid-cell center
representative_lon = sample["geometry"]["coordinates"][0]
representative_lat = sample["geometry"]["coordinates"][1]

print("\nRepresentative ERA5-Land pixel:")
print("Latitude :", representative_lat)
print("Longitude:", representative_lon)

Pincode centroid:
23.02227361979656 72.56121992376342

Representative ERA5-Land pixel:
Latitude : 22.999693294401865
Longitude: 72.60115569230976
